In [1]:
import re
import os
import json
import bisect
import warnings
import polars as pl
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import torch.nn as nn

from tqdm.notebook import tqdm
from math import ceil
from typing import *

In [2]:
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(50)
pl.Config.set_fmt_float("full")


polars.config.Config

# functions

In [3]:
def add_emer_outp_boundries(subject_sequence: pd.DataFrame):
    
    hadm_ids = subject_sequence.hadm_id.unique()
    hadm_ids = hadm_ids[~np.isnan(hadm_ids)]
    
    patient_admissions = []
    demographics = subject_sequence[subject_sequence.code.str.startswith(('RACE','GENDER'))]
    death = subject_sequence[subject_sequence.code.str.startswith('MEDS_DEATH')]
    
    for idx in hadm_ids:
        patient_admissions.append(subject_sequence[subject_sequence.seq_id == idx])
    
    processed_admissions = []
    for admission in patient_admissions:
        
        seq_id = admission.seq_id.unique()[0]

        emer = admission[admission.er_id==seq_id]
        if emer.shape[0] != 0:
            start = emer.iloc[[0]]
            start['code'] = 'EMERGENCY-START'
            start.iloc[:,9:] =  np.nan
            start['code_type'] = 'EMERGENCY-START'

            end = emer.iloc[[-1]]
            end['code'] = 'EMERGENCY-END'
            end.iloc[:,9:] =  np.nan
            end['code_type'] = 'EMERGENCY-END'
            emer = pd.concat((start,emer,end))

        hadm = admission[admission.hadm_id==seq_id]
        out = admission[admission.out_id==seq_id]

        pre_out = out[out.time < hadm.time.iloc[0]]
        if pre_out.shape[0] != 0:
            start = pre_out.iloc[[0]]
            start['code'] = 'OUTPATIENT-START'
            start.iloc[:,9:] =  np.nan
            start['code_type'] = 'OUTPATIENT-START'
            
            end = pre_out.iloc[[-1]]
            end['code'] = 'OUTPATIENT-END'
            end.iloc[:,9:] =  np.nan
            end['code_type'] = 'OUTPATIENT-END'

            pre_out = pd.concat((start,pre_out,end))

        post_out = out[out.time > hadm.time.iloc[-1]]
        if post_out.shape[0] != 0:
            start = post_out.iloc[[0]]
            start['code'] = 'OUTPATIENT-START'
            start.iloc[:,9:] =  np.nan
            start['code_type'] = 'OUTPATIENT-START'

            end = post_out.iloc[[-1]]
            end['code'] = 'OUTPATIENT-END'
            end.iloc[:,9:] =  np.nan
            end['code_type'] = 'OUTPATIENT-END'

            post_out = pd.concat((start,post_out,end))
        
        processed = pd.concat((pre_out,emer,hadm,post_out))
        processed_admissions.append(processed)
    
    return pd.concat((demographics,*processed_admissions,death))

In [4]:
def within(duration: float):
    if duration >= 0 and duration <= 7:
        return '1-W'
    elif duration > 7 and duration <= 14:
        return '2-W'
    elif duration > 14 and duration <= 21:
        return '3-W'
    elif duration > 21 and duration <= 30:
        return '1-M'
    elif duration > 30 and duration <= 60:
        return '2-M'    
    elif duration > 60 and duration <= 90:
        return '3-M' 
    elif duration > 90 and duration <= 120:
        return '4-M' 
    elif duration > 120 and duration <= 150:
        return '5-M' 
    elif duration > 150 and duration <= 180:
        return '6-M' 
    elif duration > 180 and duration <= 210:
        return '7-M' 
    elif duration > 210 and duration <= 240:
        return '8-M'
    elif duration > 240 and duration <= 270:
        return '9-M' 
    elif duration > 270 and duration <= 300:
        return '10-M'
    elif duration > 300 and duration <= 330:
        return '11-M'    
    elif duration > 330 and duration <= 360:
        return '12-M'     
    elif duration > 360:
        return '1-Y+'
    elif duration < 0:
        return '1-W'

In [5]:
def add_time_tokens(subject_sequence: pd.DataFrame):
    
    subject_sequence['time_diff'] = subject_sequence.time.diff().dt.total_seconds()/(3600*24)
    hadm_ids = subject_sequence.hadm_id.unique()
    hadm_ids = hadm_ids[~np.isnan(hadm_ids)]
    
    patient_admissions = []
    demographics = subject_sequence[subject_sequence.code.str.startswith(('RACE','GENDER'))]
    death = subject_sequence[subject_sequence.code.str.startswith('MEDS_DEATH')]
    
    for idx in hadm_ids:
        patient_admissions.append(subject_sequence[subject_sequence.seq_id == idx])
    
    processed_admissions = []
    for admission in patient_admissions:
        
        seq_id = admission.seq_id.unique()[0]
        emer = admission[admission.er_id==seq_id]
        hadm = admission[admission.hadm_id==seq_id]
        out = admission[admission.out_id==seq_id]
        pre_out = out[out.time < hadm.time.iloc[0]]
        post_out = out[out.time > hadm.time.iloc[-1]]
        
        if pre_out.shape[0] != 0:
            if pd.isna(pre_out.iloc[0].time_diff):
                pass
            else:
                time_diff = pre_out.iloc[0].time_diff
                time_period = within(time_diff)
                
                time_token = pre_out.iloc[[0]]
                time_token['code'] = 'TIME-GAP//' + time_period
                time_token.iloc[:,9:-1] =  np.nan
                time_token['numeric_value'] = time_diff
                time_token['text_value'] = time_period
                time_token['code_type'] = 'TIME-GAP'
                
                pre_out = pd.concat((time_token,pre_out))
        
        if emer.shape[0] != 0:
            if pd.isna(emer.iloc[0].time_diff):
                pass
            else:
                time_diff = emer.iloc[0].time_diff
                time_period = within(time_diff)
                
                time_token = emer.iloc[[0]]
                time_token['code'] = 'TIME-GAP//' + time_period
                time_token.iloc[:,9:-1] =  np.nan
                time_token['numeric_value'] = time_diff
                time_token['text_value'] = time_period
                time_token['code_type'] = 'TIME-GAP'
                
                emer = pd.concat((time_token,emer))
        else:
            if pd.isna(hadm.iloc[0].time_diff):
                pass
            else:
                time_diff = hadm.iloc[0].time_diff
                time_period = within(time_diff)

                time_token = hadm.iloc[[0]]
                time_token['code'] = 'TIME-GAP//' + time_period
                time_token.iloc[:,9:-1] =  np.nan
                time_token['numeric_value'] = time_diff
                time_token['text_value'] = time_period
                time_token['code_type'] = 'TIME-GAP'

                hadm = pd.concat((time_token,hadm))


        if post_out.shape[0] != 0:
            if pd.isna(post_out.iloc[0].time_diff):
                pass
            else:
                time_diff = post_out.iloc[0].time_diff
                time_period = within(time_diff)
                
                time_token = post_out.iloc[[0]]
                time_token['code'] = 'TIME-GAP//' + time_period
                time_token.iloc[:,9:-1] =  np.nan
                time_token['numeric_value'] = time_diff
                time_token['text_value'] = time_period
                time_token['code_type'] = 'TIME-GAP'
                
                post_out = pd.concat((time_token,post_out))

        
        processed = pd.concat((pre_out,emer,hadm,post_out))
        processed_admissions.append(processed)
    
    if death.shape[0] != 0:
        if death.iloc[0].time_diff > 1:
            time_diff = death.iloc[0].time_diff
            time_period = within(time_diff)
                
            time_token = death.iloc[[0]]
            time_token['code'] = 'TIME-GAP//' + time_period
            time_token.iloc[:,9:-1] =  np.nan
            time_token['numeric_value'] = time_diff
            time_token['text_value'] = time_period
            time_token['code_type'] = 'TIME-GAP'
                
            death = pd.concat((time_token,death))
            
            
    
    return pd.concat((demographics,*processed_admissions,death),)

In [6]:
def z_score_normalize(
    shard_fp: str,
    metadata_fp: str,
    code_col: str = "code",
    value_col: str = "numeric_value",
) -> pd.DataFrame:
    """
    Read a MEDS shard and normalization metadata, z-score normalize per-code,
    overwrite `numeric_value` in place, and return the normalized shard (as a DataFrame).

    Inputs
    ------
    shard_fp: path to a MEDS shard parquet (e.g., ".../data/train/123.parquet")
    metadata_fp: path to metadata parquet produced from aggregates/fit (per-code stats)
                 Expected to contain either:
                   - columns: ["code", "values/mean", "values/std"], OR
                   - columns: ["code", "values/n_occurrences", "values/sum", "values/sum_sqd"]
    code_col: column name for codes in shard/metadata (default "code")
    value_col: numeric column to normalize (default "numeric_value")

    Returns
    -------
    pandas.DataFrame with `numeric_value` normalized in place.
    """

    # Load shard & metadata
    df = pd.read_parquet(shard_fp)
    meta = pd.read_parquet(metadata_fp)

    # Keep only needed columns from metadata
    cols = set(meta.columns)

    # Case A: metadata already has mean/std
    if {"values/mean", "values/std"}.issubset(cols):
        meta_use = meta[[code_col, "values/mean", "values/std"]].rename(
            columns={"values/mean": "_mu", "values/std": "_sd"}
        )
    # Case B: derive mean/std from counts & sums
    elif {"values/n_occurrences", "values/sum", "values/sum_sqd"}.issubset(cols):
        m = meta[[code_col, "values/n_occurrences", "values/sum", "values/sum_sqd"]].copy()
        m = m.rename(
            columns={
                "values/n_occurrences": "_n",
                "values/sum": "_sum",
                "values/sum_sqd": "_sum2",
            }
        )
        # mean = sum / n ; var = (sum2/n) - mean^2  (unbiased vs. population: use population here to match typical pipeline)
        m["_mu"] = m["_sum"] / m["_n"].replace(0, np.nan)
        m["_var"] = (m["_sum2"] / m["_n"].replace(0, np.nan)) - (m["_mu"] ** 2)
        m["_sd"] = np.sqrt(m["_var"].clip(lower=0.0))
        meta_use = m[[code_col, "_mu", "_sd"]]
    else:
        raise ValueError(
            "Metadata missing required columns. Provide mean/std or n/sum/sum_sqd."
        )

    # Join per-code stats to shard
    df = df.merge(meta_use, on=code_col, how="left")

    # Normalize in place: z = (x - mu) / sd ; leave values untouched if sd==0 or stats missing
    x = df[value_col].astype("float64")
    mu = df["_mu"]
    sd = df["_sd"]

    can_norm = (~x.isna()) & (~mu.isna()) & (~sd.isna()) & (sd > 0)
    df.loc[can_norm, value_col] = (x[can_norm] - mu[can_norm]) / sd[can_norm]

    # Drop helper columns
    df = df.drop(columns=["_mu", "_sd"], errors="ignore")

    return df

In [7]:
def minmax_scale(
    shard_fp: str,
    metadata_fp: str,
    code_col: str = "code",
    value_col: str = "numeric_value",
#     new_col: str = None,
) -> pd.DataFrame:
    """
    Apply per-code min–max scaling to a MEDS shard using metadata.

    Each numeric value is scaled as: (x - min) / (max - min).

    Parameters
    ----------
    shard_fp : str
        Path to a MEDS shard parquet (e.g., ".../data/train/123.parquet").
    metadata_fp : str
        Path to metadata parquet with 'values/min' and 'values/max' columns.
    code_col : str, default="code"
        Column name for codes in shard/metadata.
    value_col : str, default="numeric_value"
        Numeric column to scale.
    new_col : str, optional
        If provided, scaled values are written to this new column.
        If None, the function overwrites `value_col` in place.

    Returns
    -------
    df : pd.DataFrame
        Shard with scaled values (either in-place or in a new column).
    """

    # Load data
    df = pd.read_parquet(shard_fp)
    meta = pd.read_parquet(metadata_fp)[[code_col, "values/min", "values/max"]]

    # Rename to simplify join
    meta = meta.rename(columns={"values/min": "_min", "values/max": "_max"})

    # Merge stats onto shard
    df = df.merge(meta, on=code_col, how="left")

    # Scale values
    x = df[value_col].astype("float64")
    minv = df["_min"]
    maxv = df["_max"]

    denom = (maxv - minv).replace(0, 1)  # avoid divide-by-zero
    scaled = (x - minv) / denom

#     if new_col is None:
    df[value_col] = scaled
#     else:
#         df[new_col] = scaled

    # Drop helper cols
    df = df.drop(columns=["_min", "_max"])

    return df

In [8]:
def normalize(shard_fp: str,
              metadata_fp: str,
              code_col: str = "code",
              mode: str = 'z_score',
              value_col: str = "numeric_value",
             ) -> pd.DataFrame:
    
    assert mode in ['z_score','min/max']
    
    if mode == 'z_score':
        shard = z_score_normalize(shard_fp=shard_fp,
                                  metadata_fp=metadata_fp,
                                  code_col=code_col,
                                  value_col=value_col)
    elif mode == 'min/max':
        shard = minmax_scale(shard_fp=shard_fp,
                             metadata_fp=metadata_fp,
                             code_col=code_col,
                             value_col=value_col)
    
    return shard

In [9]:
import os, glob, yaml
import numpy as np
import pandas as pd

def build_decile_yaml(meds_root, out_yaml, probs=(0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9)):
    paths = glob.glob(os.path.join(meds_root, "data", "**", "*.parquet"), recursive=True)
    if not paths:
        raise RuntimeError(f"No shards under {meds_root}/data")
    dfs = [pd.read_parquet(p, columns=["code", "numeric_value"]) for p in paths]
    df = pd.concat(dfs, ignore_index=True).dropna(subset=["numeric_value"])
    out = {}
    for code, g in df.groupby("code")["numeric_value"]:
        v = g.to_numpy()
        if np.unique(v).size < 2:
            out[code] = {"const": float(v[0]) if v.size else 0.0}  # single right endpoint -> two bins
            continue
        qs = np.quantile(v, probs)
        # enforce strict ascending by removing duplicates
        edges = []
        for x in qs:
            x = float(x)
            if not edges or x > edges[-1]:
                edges.append(x)
        # name keys to preserve order; values strictly ascending
        bins_dict = {f"q{p:0.2f}": val for p, val in zip(probs, edges)}
        out[code] = bins_dict
    
    with open(out_yaml, "w") as f:
        yaml.safe_dump(out, f, sort_keys=False)
    return out

In [10]:
import polars as pl
from pathlib import Path

def split_fused_code_interleaved(df: pl.DataFrame) -> pl.DataFrame:
    # preserve current row order
    dfi = df.with_row_index("_row")

    c = pl.col("code")
    has_bin = c.str.contains(r"//\[")
    base = c.str.replace(r"//\[.*\)$", "")
    bin_  = c.str.extract(r"(//\[.*\))$", 1)
    bin_code = pl.concat_str([pl.lit("VALUE"), pl.lit('//'), base, bin_])

    # base events (always 1 per input row)
    base_rows = (
        dfi.with_columns([
            pl.when(has_bin).then(base).otherwise(c).alias("code"),
            pl.lit(0).alias("__kind")  # for ordering base before bin
        ])
    )

    # bin token events (only for rows that had a bin suffix)
    bin_rows = (
        dfi.filter(has_bin)
           .with_columns([
               bin_code.alias("code"),
               pl.lit(None).alias("numeric_value"),
               pl.lit(1).alias("__kind")
           ])
    )

    # stack and interleave by original row index, base first then bin
    out = pl.concat([base_rows, bin_rows], how="diagonal_relaxed") \
            .sort(["_row", "__kind"]) \
            .drop(["_row", "__kind"])

    return out


def transform_shard(in_fp: str, out_fp: str):
    df = pl.read_parquet(in_fp)
    out = split_fused_code_interleaved(df)
    Path(out_fp).parent.mkdir(parents=True, exist_ok=True)
    out.write_parquet(out_fp)
#     return out

# Tokenizer building

In [11]:
import tokenizers
import transformers 

In [12]:
data_path = os.path.join('..','data','meds_normalized','data','train')
metadata_path = os.path.join('..','data','meds_normalized','metadata')

shard = pl.read_parquet(os.path.join(data_path,'1.parquet'))
metadata = pl.read_parquet(os.path.join(metadata_path,'codes.parquet'))

In [13]:
from __future__ import annotations
from pathlib import Path
from typing import Iterable, Dict, List, Optional, Union
import json
import polars as pl

class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str | Path,
        special_tokens: Optional[Iterable[str]] = None,
        force_special_ids: bool = True,  # pin [PAD]=0 etc.
    ):
        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()
        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        vocab_list: List[str] = []
        special_tokens = list(special_tokens)

        if force_special_ids:
            for tok in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]:
                if tok in special_tokens and tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
                elif tok in special_tokens and tok in seen:

                    vocab_list.append(tok)
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)
        else:
            for tok in special_tokens:
                if tok not in seen:
                    vocab_list.append(tok)
                    seen.add(tok)


        vocab_list.extend(unique_codes)


        self.id2code: List[str] = vocab_list
        self.code2id: Dict[str, int] = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.vocab_size: int = len(self.id2code)

        self.pad_token  = "[PAD]" if "[PAD]" in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]" if "[CLS]" in self.code2id else None
        self.unk_token  = "[UNK]" if "[UNK]" in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else 0
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None


        type_set = set()
        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        types_sorted = sorted(t for t in type_set if t not in ("[PAD]",))
        self.type2id: Dict[str, int] = {"[PAD]": 0}
        next_id = 1
        for sp in ["[MASK]", "[CLS]", "[UNK]"]:
            if sp in type_set:
                self.type2id[sp] = next_id; next_id += 1
        for t in types_sorted:
            if t not in self.type2id:
                self.type2id[t] = next_id
                next_id += 1

        self._code2id_df = pl.DataFrame({"code": self.id2code,
                                         "input_id": list(range(self.vocab_size))}) \
                               .with_columns(pl.col("code").cast(pl.Categorical))
        self._type2id_df = pl.DataFrame({"code_type": list(self.type2id.keys()),
                                         "type_id":   list(self.type2id.values())}) \
                               .with_columns(pl.col("code_type").cast(pl.Categorical))


    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        if self.unk_id is not None:
            fallback = self.unk_id
        else:
            fallback = self.pad_id if self.pad_id is not None else 0
        return [get(c, fallback) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out

    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        obj = {
            "id2code": self.id2code,
            "special_tokens": [t for t in ["[PAD]", "[MASK]", "[CLS]", "[UNK]"] if t in self.code2id],
            "type2id": self.type2id,
            "pad_id": self.pad_id,
            "mask_id": self.mask_id,
            "cls_id": self.cls_id,
            "unk_id": self.unk_id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)

    @classmethod
    def load(cls, path: str | Path) -> "Tokenizer":
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls) 

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = obj.get("pad_id", tok.code2id.get("[PAD]", 0))
        tok.mask_id = obj.get("mask_id", tok.code2id.get("[MASK]")) if "[MASK]" in tok.code2id else None
        tok.cls_id  = obj.get("cls_id", tok.code2id.get("[CLS]"))   if "[CLS]" in tok.code2id else None
        tok.unk_id  = obj.get("unk_id", tok.code2id.get("[UNK]"))   if "[UNK]" in tok.code2id else None

        # Rebuild Polars lookup frames
        tok._code2id_df = pl.DataFrame({"code": tok.id2code,
                                        "input_id": list(range(tok.vocab_size))}) \
                              .with_columns(pl.col("code").cast(pl.Categorical))
        tok._type2id_df = pl.DataFrame({"code_type": list(tok.type2id.keys()),
                                        "type_id":   list(tok.type2id.values())}) \
                              .with_columns(pl.col("code_type").cast(pl.Categorical))
        return tok

    @property
    def code2id_df(self) -> pl.DataFrame:
        return self._code2id_df

    @property
    def type2id_df(self) -> pl.DataFrame:
        return self._type2id_df

In [14]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


In [15]:
from __future__ import annotations
from math import ceil
from typing import Iterable, Dict, List, Optional, Union, Any, Literal
import polars as pl
import torch

class SequencesGenerator:


    def __init__(
        self,
        tokenizer_path: str,
        chunk_length: int = 1024,
        overlap: int = 128,
        return_numeric: bool = False,
        return_text: bool = False,
    ):

        self.tokenizer = Tokenizer.load(tokenizer_path)
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text

    def encode_sequence(
        self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
    ) -> Dict[str, Union[List[int], List[float], List[str]]]:
        """
        Vectorized build of:
          input_ids, attention_mask, visit_ids, stage_ids, type_ids
          + optional numeric/text streams (+ masks)
        """
        df = timeline

        # --- visit_ids (dense per-timeline integer ids starting at 1) ---
        # If seq_id exists -> categorical codes as stable ints; otherwise zeros.
        if "seq_id" in df.columns:
        # unique stable integer ids starting at 1
            uniq = df.select(pl.col("seq_id")).unique(maintain_order=True)
            uniq = uniq.with_row_count(name="visit_ids_raw")  # 0..K-1
            df = df.join(uniq, on="seq_id", how="left").with_columns(
                (pl.col("visit_ids_raw") ).alias("visit_id").fill_null(0)
            ).drop("visit_ids_raw")
        else:
            df = df.with_columns(pl.lit(0).alias("visit_id"))

        # --- stage_ids (first non-null of stage cols) ---
        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        present_stages = [c for c in stage_cols if c in df.columns]
        if present_stages:
            # Compute stage_id = 1..len(present_stages) or 0 if all null
            # We scan in the stage_cols order and pick the first non-null
            expr = pl.lit(0)
            for i, col in enumerate(present_stages, start=1):
                expr = pl.when(expr.eq(0) & pl.col(col).is_not_null()).then(i).otherwise(expr)
            df = df.with_columns(expr.alias("stage_id"))
        else:
            df = df.with_columns(pl.lit(0).alias("stage_id"))

        # --- map code_type -> type_id (PAD type=0 already in tokenizer) ---
        df = df.join(
            self.tokenizer.type2id_df,
            on=pl.col("code_type").cast(pl.Categorical),
            how="left",
        ).with_columns(pl.col("type_id").fill_null(0))

        # --- map code -> input_id (PAD=0, UNK fallback handled downstream) ---
        df = df.join(
            self.tokenizer.code2id_df,
            on=pl.col("code").cast(pl.Categorical),
            how="left",
        )

        # Fill unknown codes to UNK (or PAD if no UNK)
        unk_id = self.tokenizer.unk_id if self.tokenizer.unk_id is not None else self.tokenizer.pad_id or 0
        df = df.with_columns(pl.col("input_id").fill_null(unk_id))

        # --- TIME-GAP tokens zero out visit_id & stage_id (patient-agnostic) ---
        df = df.with_columns(
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("visit_id"))
              .alias("visit_id"),
            pl.when(pl.col("code").str.starts_with("TIME-GAP//"))
              .then(0)
              .otherwise(pl.col("stage_id"))
              .alias("stage_id"),
        )

        # --- Add [CLS] row if requested ---
        if add_cls and (self.tokenizer.cls_token is not None):
            cls_row = {
                "code": self.tokenizer.cls_token,
                "code_type": "[CLS]",
                "visit_id": 0,
                "stage_id": 0,
                "type_id": self.tokenizer.type2id.get("[CLS]", 0),
                "input_id": self.tokenizer.cls_id,
            }
            # Optional numeric/text placeholders
            if self.return_numeric:
                cls_row["numeric_value"] = None
            if self.return_text:
                cls_row["text_value"] = None

            df = pl.concat([pl.DataFrame([cls_row]), df], how="vertical_relaxed")

        # --- extract arrays ---
        input_ids = df.get_column("input_id").cast(pl.Int64).to_list()
        type_ids = df.get_column("type_id").cast(pl.Int64).to_list()
        visit_ids = df.get_column("visit_id").cast(pl.Int64).to_list()
        stage_ids = df.get_column("stage_id").cast(pl.Int64).to_list()
        attention_mask = [1] * len(input_ids)

        # --- optional value streams (numeric/text) ---
        value_payload = self._build_value_streams(
            df=df,
            max_length=max_length,
            pad_to_max=pad_to_max,
            truncation=truncation,
        )

        # --- truncate/pad core streams in one go ---
        input_ids      = self._truncate(input_ids,      max_length, truncation)
        type_ids       = self._truncate(type_ids,       max_length, truncation)
        visit_ids      = self._truncate(visit_ids,      max_length, truncation)
        stage_ids      = self._truncate(stage_ids,      max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids      = input_ids + [pad_id] * pad_len
            type_ids       = type_ids + [0] * pad_len
            visit_ids      = visit_ids + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
            attention_mask = attention_mask + [0] * pad_len

        out = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "visit_ids": visit_ids,
            "stage_ids": stage_ids,
            "type_ids": type_ids,
        }
        out.update(value_payload)
        return out

    def get_overlapped_chunks(
        self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True,
    ) -> List[Dict[str, List[Any]]]:
        """
        Sliding-window chunking with optional [CLS] per chunk and padding.
        """
        if chunk_length is None or overlap is None:
            chunk_length = self.chunk_length
            overlap = self.overlap

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask"):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])
        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = max(1, payload - overlap)

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {
            "input_ids": self.tokenizer.cls_id if self.tokenizer.cls_id is not None else (self.tokenizer.pad_id or 0),
            "attention_mask": 1,
            "visit_ids": 0,
            "stage_ids": 0,
            "type_ids": self.tokenizer.type2id.get("[CLS]", 0),
            "numeric_values": 0.0,
            "numeric_mask": 0,
            "text_values": "",
            "text_mask": 0,
        }

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: list(timeline[k][start:end]) for k in fields if k in timeline}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)
        return chunks


    def _build_value_streams(
        self,
        df: pl.DataFrame,
        max_length: Optional[int],
        pad_to_max: bool,
        truncation: Literal["head", "tail"],
    ) -> Dict[str, List[Any]]:
        out: Dict[str, List[Any]] = {}
        # Numeric stream
        if self.return_numeric:
            if "numeric_value" in df.columns:
                vals = df.get_column("numeric_value").to_list()
            else:
                vals = [None] * df.height
            num_mask = [1 if (v is not None) else 0 for v in vals]
            vals = [0.0 if v is None else float(v) for v in vals]

            vals = self._truncate(vals, max_length, truncation)
            num_mask = self._truncate(num_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(vals) < max_length:
                pad_len = max_length - len(vals)
                vals += [0.0] * pad_len
                num_mask += [0] * pad_len

            out["numeric_values"] = vals
            out["numeric_mask"] = num_mask

        # Text stream
        if self.return_text:
            if "text_value" in df.columns:
                txt = df.get_column("text_value").to_list()
            else:
                txt = [None] * df.height
            txt = [("" if (t is None or str(t) == "___") else str(t)) for t in txt]
            txt_mask = [1 if (t != "") else 0 for t in txt]

            txt = self._truncate(txt, max_length, truncation)
            txt_mask = self._truncate(txt_mask, max_length, truncation)

            if pad_to_max and max_length is not None and len(txt) < max_length:
                pad_len = max_length - len(txt)
                txt += [""] * pad_len
                txt_mask += [0] * pad_len

            out["text_values"] = txt
            out["text_mask"] = txt_mask

        return out

    @staticmethod
    def _truncate(seq: List[Any], max_length: Optional[int], truncation: str) -> List[Any]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]

    @staticmethod
    def _pad_list(lst: List[Any], pad_len: int, pad_value: Any) -> List[Any]:
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len

In [16]:
class EHRPretrainDataset(Dataset):
    def __init__(self,
                 data_path: str,
                 data_idx_path: str,
                 seq_generator: SequencesGenerator) -> None:
        
        self.data_path = data_path
        self.seq_generator = seq_generator
        
        
        data_idx =  pl.scan_parquet(data_idx_path).collect()
        self.data_idx, self.cum, self.subj = self._get_chunks_count(data_idx= data_idx,
                                                                    chunk_length=self.seq_generator.chunk_length,
                                                                    overlap=self.seq_generator.overlap)
        
    def __len__(self) -> int:
        return self.cum[-1]


    
    def __getitem__(self,
                    index: int):
        

        subject_id, chunk_id = self._get_chunk_at_idx(idx=index,
                                                      cumm_sum=self.cum,
                                                      subjects=self.subj)
        timeline = self._read_timeline(subject_id)
        timeline_encoded = self.seq_generator.encode_sequence(timeline=timeline)
        chunks = self.seq_generator.get_overlapped_chunks(timeline= timeline_encoded,
                                                          chunk_length= self.seq_generator.chunk_length,
                                                          overlap=self.seq_generator.overlap)
        
#         chunk = self.seq_generator._return_tensors(chunks[chunk_id])
        return chunks[chunk_id]
    

    def _build_dataset_index(self,
                             data_path, 
                             subject_col="subject_id") -> pl.DataFrame:
        pieces = []
        for p in os.listdir(data_path):
            df = (pl.scan_parquet(os.path.join(data_path,p)).select(subject_col).collect()
                    .group_by(subject_col)
                    .len()
                    .rename({"len": "n_events"})
                 )
            df = df.with_columns(pl.lit(str(p)).alias("shard"))  # optional
            pieces.append(df)


        df = (pl.concat(pieces, how="vertical")
                  .group_by([subject_col, "shard"])
                  .agg(pl.col("n_events").sum())
                  .rename({subject_col:"subject_id"})).sort('subject_id')

        df = df.filter(pl.col('n_events') >3)

        return df
    
    
    def _read_timeline(self,
                       subject_id:int) -> pl.DataFrame:
        
        shard = self.data_idx.filter(pl.col('subject_id') == subject_id)['shard'][0]

        data = pl.scan_parquet(os.path.join(self.data_path,shard),parallel='auto').select(
                                            ['subject_id','seq_id','out_id','er_id','hadm_id', 
                                             'icustay_id','time','code','numeric_value','code_type',
                                             'text_value']).filter(
                                              pl.col('subject_id') == subject_id).collect()
        return data

    
    def _get_chunks_count(self,
                          data_idx: pl.DataFrame,
                          chunk_length: int,
                          overlap: int):
        payload  = chunk_length - 1
        step     = payload - overlap

        data_idx = data_idx.with_columns(
            pl.col("n_events")
              .map_elements(lambda n: 1 if n<=payload else ceil((n-payload)/step)+1)
              .alias("n_chunks")
        )
        data_idx = data_idx.with_columns(
            pl.col("n_chunks").cum_sum().alias("cum_chunks")
        )

        cum  = data_idx["cum_chunks"]   
        subj = data_idx["subject_id"]
        shards = data_idx['shard']
        return data_idx, cum, subj

    def _get_chunk_at_idx(self,
                          cumm_sum: list,
                          subjects: list,
                          idx: int) -> Tuple[int,int,str]:
        i = bisect.bisect_right(cumm_sum, idx)
        left = cumm_sum[i-1] if i > 0 else 0
        return subjects[i], idx - left

In [17]:
seq_gen = SequencesGenerator(tokenizer_path='./toy_label/vocab.json',
                             chunk_length=1024,
                             overlap=128,
                             return_numeric=False,
                             return_text=False)

In [18]:
dataset = EHRPretrainDataset(data_idx_path='./dataset_idx.parquet',
                             data_path='../data/meds_normalized/data/train/',
                             seq_generator=seq_gen)

In [19]:
class MLMDataCollator:
    def __init__(
        self,
        tokenizer,
        protected_tokens: List[str],
        mask_prob: float = 0.15,
        replace_prob: float = 0.80,
        random_prob: float = 0.10,
    ) -> None:

        self.tokenizer = tokenizer
        self.mask_prob = mask_prob
        self.replace_prob = replace_prob
        self.random_prob = random_prob
        self.protected_ids = self._build_protected_ids(protected_tokens)
        if tokenizer.mask_id is None:
            raise ValueError("Tokenizer must define a [MASK] token/id.")

    def __call__(self, batch: List[Union[Dict, List[Dict]]]) -> Dict[str, torch.Tensor]:
        chunks = self._flatten(batch)
        out = self._stack(chunks)
        masked_ids, labels = self._mask_batch(out["input_ids"], out["attention_mask"])
        out["input_ids"] = masked_ids
        out["labels"] = labels
        return out


    def _flatten(self, batch) -> List[Dict]:
        out: List[Dict] = []
        for item in batch:
            if isinstance(item, dict):
                out.append(item)
            elif isinstance(item, (list, tuple)):
                out.extend(item)
            else:
                raise TypeError(f"Unexpected item type: {type(item)}")
        if not out:
            raise ValueError("Empty batch after normalization.")
        return out

    def _stack(self, chunks: List[Dict]) -> Dict[str, torch.Tensor]:
        keys = list(chunks[0].keys())
        out = {}
        for k in keys:
            # Skip known non-numeric or variable-shaped fields
            if k in ("text_values",):  # add others you don't want to collate
                continue

            # Replace Nones with safe defaults
            seq_list = []
            for c in chunks:
                v = c[k]
                if isinstance(v, list):
                    v = [0 if x is None else x for x in v]  # 0 for ints/floats
                elif v is None:
                    # single value case (shouldn't happen for sequences, but guard anyway)
                    v = 0
                seq_list.append(torch.as_tensor(v))
            out[k] = torch.stack(seq_list, 0)
        return out

    def _build_protected_ids(self, protected_tokens: List[str]) -> torch.BoolTensor:

        ids = set()
        for tok in protected_tokens:
            if tok in self.tokenizer.code2id:
                ids.add(self.tokenizer.code2id[tok])
        mask = torch.zeros(self.tokenizer.vocab_size, dtype=torch.bool)
        for i in ids:
            mask[i] = True
        return mask

    def _mask_batch(
        self,
        input_ids: torch.Tensor,      # (B, L)
        attention_mask: torch.Tensor  # (B, L)
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Vectorized BERT-style masking:
        • 80% -> [MASK]
        • 10% -> random token (not protected)
        • 10% -> keep original
        Returns masked_input_ids, labels.
        """
        device = input_ids.device
        prot = self.protected_ids.to(device)
        eligible = attention_mask.bool() & (~prot[input_ids])

        sample = torch.rand_like(input_ids.float()) < self.mask_prob
        to_mask = eligible & sample

        masked = input_ids.clone()
        labels = torch.full_like(input_ids, -100)
        labels[to_mask] = input_ids[to_mask]

        r = torch.rand_like(input_ids.float())
        to_mask80 = to_mask & (r < self.replace_prob)
        to_rand10 = to_mask & (r >= self.replace_prob) & (r < self.replace_prob + self.random_prob)

        # 80% -> [MASK]
        masked[to_mask80] = self.tokenizer.mask_id

        # 10% -> random allowed token
        allowed = (~prot).nonzero(as_tuple=False).squeeze(1).to(device)
        if allowed.numel() == 0:
            allowed = torch.arange(self.tokenizer.vocab_size, device=device)
        if to_rand10.any():
            rand_ids = allowed[torch.randint(0, allowed.numel(), (to_rand10.sum(),), device=device)]
            masked[to_rand10] = rand_ids
        return masked, labels

In [20]:
protected_tokens = [
    "[PAD]", "[CLS]", "[MASK]",
    "OUTPATIENT-START","OUTPATIENT-END",
    "EMERGENCY-START","EMERGENCY-END",
    "ADMISSION-AT-HOSPITAL","ADMISSION-AT-ICU",
    "DISCHARGE-FROM-HOSPITAL","DISCHARGE-FROM-ICU",
]

collate_fn = MLMDataCollator(
    tokenizer=seq_gen.tokenizer,
    protected_tokens=protected_tokens,
    mask_prob=0.15, replace_prob=0.80, random_prob=0.10)



dl = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [21]:
# IMPORTANT
MAX_VISITS = 101 # account for th padding
VOCAB_SIZE = 47377

import torch
import torch.nn as nn

class EHREmbeddings(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embedding_size: int,
        pad_token_id: int = 0,
        type_vocab_size: int = 28,
        visit_vocab_size: int = 102,
        stage_vocab_size: int = 5,
        dropout: float = 0.1
    ):
        super().__init__()
        self.tok_emb   = nn.Embedding(vocab_size, embedding_size, padding_idx=pad_token_id)
        self.type_emb  = nn.Embedding(type_vocab_size,  embedding_size, padding_idx=pad_token_id)
        self.visit_emb = nn.Embedding(visit_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.stage_emb = nn.Embedding(stage_vocab_size, embedding_size, padding_idx=pad_token_id)
        self.norm      = nn.LayerNorm(embedding_size)
        self.drop      = nn.Dropout(dropout)

    def encode(self, input_ids, type_ids, visit_ids, stage_ids):
        x = self.tok_emb(input_ids.long())
        x = x + self.type_emb(type_ids.long())
        x = x + self.visit_emb(visit_ids.long())
        x = x + self.stage_emb(stage_ids.long())
        return self.drop(self.norm(x))


    def forward(self, input_ids=None, 
                token_type_ids=None, 
                inputs_embeds=None,
                **kwargs):
        if inputs_embeds is not None:
            return inputs_embeds
        return self.drop(self.norm(self.tok_emb(input_ids.long())))

In [22]:
# x = next(iter(dl))
# model = RoFormerEHR(cfg)


In [23]:
# model.backbone(x)

In [24]:
# model = RoFormerForMaskedLM(config=cfg)
# model.

In [25]:
# x['input_ids'].shape

In [26]:
import torch
import torch.nn as nn
import lightning as l
from torch.optim import AdamW
from transformers import RoFormerConfig, RoFormerForMaskedLM, get_linear_schedule_with_warmup


cfg = RoFormerConfig(
    vocab_size=VOCAB_SIZE,
    hidden_size=768,
    num_hidden_layers=12,
    num_attention_heads=12,
    intermediate_size=3072,
    max_position_embeddings=1024,
    pad_token_id=0,
    type_vocab_size= 28,
    visit_vocab_size= 102,
    stage_vocab_size= 5,
    rotary_pct=1.0,                 # full rotary on Q/K
    attention_probs_dropout_prob=0.1,
    hidden_dropout_prob=0.1,)



# ----- Lightning module with model inside -----
class MLMPretraining(l.LightningModule):
    """
    One-class trainer:
      - builds RoFormerForMaskedLM
      - swaps in EHREmbeddings
      - ties decoder to token embedding
      - trains with MLM labels (ignore_index = -100)
    Batch contract (keys): input_ids, attention_mask, type_ids, visit_ids, stage_ids, labels
    """
    def __init__(
        self,
        config,
        lr: float = 2e-5,
        wd: float = 0.001,
        max_epochs: int = 100,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.save_hyperparameters()



        self.backbone = RoFormerForMaskedLM(config)


        ehr_emb = EHREmbeddings(
            vocab_size=config.vocab_size, 
            embedding_size=config.embedding_size, 
            pad_token_id=config.pad_token_id,
            type_vocab_size=config.type_vocab_size, 
            visit_vocab_size=config.visit_vocab_size,
            stage_vocab_size=config.stage_vocab_size, 
            dropout=dropout)
        
        self.backbone.roformer.embeddings = ehr_emb
        self.backbone.cls.predictions.decoder.weight = self.backbone.roformer.embeddings.tok_emb.weight

        self.lr = lr
        self.wd = wd
        self.max_epochs = max_epochs



    def forward(self, 
                input_ids, 
                attention_mask, 
                type_ids, 
                visit_ids, 
                stage_ids, 
                labels=None):
        
        inputs_embeds = self.backbone.roformer.embeddings.encode(
        input_ids=input_ids,
        type_ids=type_ids,
        visit_ids=visit_ids,
        stage_ids=stage_ids)
        return self.backbone(inputs_embeds=inputs_embeds,
                             attention_mask=attention_mask,
                             labels=labels)


    def training_step(self, batch, batch_idx):
        out = self.forward(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"],
                            type_ids=batch["type_ids"],
                            visit_ids=batch["visit_ids"],
                            stage_ids=batch["stage_ids"],
                            labels=batch["labels"])
        
        loss = out.loss
#         self.log("train/loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

#     def validation_step(self, batch, batch_idx):
#         out = self(
#             input_ids=batch["input_ids"],
#             attention_mask=batch["attention_mask"],
#             type_ids=batch["type_ids"],
#             visit_ids=batch["visit_ids"],
#             stage_ids=batch["stage_ids"],
#             labels=batch["labels"],
#         )
#         val_loss = out.loss
#         self.log("val/loss", val_loss, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):


        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr,weight_decay=self.wd)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer,
                                                         eta_min=0,
                                                         T_max=self.max_epochs
                                                         )
        
        return {'optimizer': optimizer,
                'lr_scheduler': scheduler}

In [27]:
model = MLMPretraining(config=cfg)
x = next(iter(dl))

RoFormerForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


In [31]:
# z = model(input_ids=x['input_ids'],
#               attention_mask=x['attention_mask'],
#               type_ids=x['type_ids'],
#               visit_ids=x['visit_ids'],
#               stage_ids=x['stage_ids'],
#               labels=None)

In [30]:
# from lightning.pytorch.strategies import DDPStrategy
# trainer = l.Trainer(precision=16,max_epochs=4,num_nodes=1,devices=1, strategy=DDPStrategy(find_unused_parameters=False, process_group_backend="nccl"),accelerator="gpu")
# torch.set_float32_matmul_precision('high')

In [32]:
# trainer.fit(model=model,train_dataloaders=dl)

In [33]:
# seq_gen.tokenizer.pad_token

In [34]:
# pad_token_id

In [61]:
from __future__ import annotations
from pathlib import Path
from typing import Iterable, Dict, List, Optional
import polars as pl
# We'll prefer polars (fast with parquet), but fall back to pandas if needed.



class Tokenizer:
    def __init__(
        self,
        codes_parquet_fp: str | Path,
        special_tokens: Optional[Iterable[str]] = None,
        include_missing_specials_only: bool = True,
    ):

        if special_tokens is None:
            special_tokens = ["[PAD]", "[MASK]", "[CLS]", "[UNK]"]


        # Read just the 'code' column using polars (fast).
        df_codes = pl.read_parquet(str(codes_parquet_fp), columns=["code"])
        base_codes = df_codes.get_column("code").to_list()

        seen = set()
        unique_codes = []
        for c in base_codes:
            if c not in seen:
                unique_codes.append(c)
                seen.add(c)

        if include_missing_specials_only:
            specials_to_add = [s for s in special_tokens if s not in seen]
        else:

            specials_to_add = list(special_tokens)

        self.special_tokens = []
        vocab_list = []


        for s in specials_to_add:
            if s not in seen:
                vocab_list.append(s)
                self.special_tokens.append(s)
                seen.add(s)
            else:
                self.special_tokens.append(s)


        vocab_list.extend(unique_codes)
        
    
        self.code2id = {tok: idx for idx, tok in enumerate(vocab_list)}
        self.id2code = vocab_list # id is identocal to index

        self.vocab_size: int = len(self.id2code)                  
        assert len(self.code2id) == self.vocab_size, "vocab mismatch / duplicates slipped in"
        
        
        self.pad_token  = "[PAD]"  if "[PAD]"  in self.code2id else None
        self.mask_token = "[MASK]" if "[MASK]" in self.code2id else None
        self.cls_token  = "[CLS]"  if "[CLS]"  in self.code2id else None
        self.unk_token  = "[UNK]"  if "[UNK]"  in self.code2id else None

        self.pad_id  = self.code2id[self.pad_token]  if self.pad_token  else None
        self.mask_id = self.code2id[self.mask_token] if self.mask_token else None
        self.cls_id  = self.code2id[self.cls_token]  if self.cls_token  else None
        self.unk_id  = self.code2id[self.unk_token]  if self.unk_token  else None
        
        self.type2id = {}
        type_set = set()

        for tok in self.id2code:
            prefix = tok.split("//", 1)[0]
            type_set.add(prefix)

        # Assign IDs starting at 1
        for i, t in enumerate(sorted(type_set), start=1):
            self.type2id[t] = i
        

    def encode(self, codes: Iterable[str]) -> List[int]:
        get = self.code2id.get
        unk = self.unk_id if self.unk_id is not None else -1
        return [get(c, unk) for c in codes]

    def decode(self, ids: Iterable[int]) -> List[str]:
        out = []
        for i in ids:
            if 0 <= i < self.vocab_size:
                out.append(self.id2code[i])
            else:
                out.append(self.unk_token or "[UNK]")
        return out
    



    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        obj = {
            "id2code": self.id2code,
            "special_tokens": self.special_tokens,
            "type2id": self.type2id,
        }
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)    

    @classmethod
    def load(cls, path: str | Path) -> "Tokenizer":
        """Load tokenizer from a saved vocab/config file."""
        path = Path(path)
        with open(path) as f:
            obj = json.load(f)

        tok = cls.__new__(cls)  # create empty instance without calling __init__

        tok.id2code = obj["id2code"]
        tok.code2id = {tok_: i for i, tok_ in enumerate(tok.id2code)}
        tok.vocab_size = len(tok.id2code)

        tok.special_tokens = obj.get("special_tokens", [])
        tok.type2id = obj.get("type2id", {})

        # Rebuild ids for special tokens (PAD/MASK/CLS/UNK) if present
        tok.pad_token  = "[PAD]" if "[PAD]" in tok.code2id else None
        tok.mask_token = "[MASK]" if "[MASK]" in tok.code2id else None
        tok.cls_token  = "[CLS]" if "[CLS]" in tok.code2id else None
        tok.unk_token  = "[UNK]" if "[UNK]" in tok.code2id else None

        tok.pad_id  = tok.code2id[tok.pad_token] if tok.pad_token else None
        tok.mask_id = tok.code2id[tok.mask_token] if tok.mask_token else None
        tok.cls_id  = tok.code2id[tok.cls_token] if tok.cls_token else None
        tok.unk_id  = tok.code2id[tok.unk_token] if tok.unk_token else None

        return tok 

In [62]:
class SequencesGenerator(object):
    def __init__(self,
                tokenizer_path:str,
                chunk_length:int =  1024,
                overlap: int = 128,
                return_numeric: bool = False,
                return_text: bool = False):
        
        self.tokenizer = Tokenizer.load(tokenizer_path)
        self.chunk_length = chunk_length
        self.overlap = overlap
        self.return_numeric = return_numeric
        self.return_text = return_text
    
    def encode_sequence(self,
        timeline: pl.DataFrame,
        max_length: Optional[int] = None,
        pad_to_max: bool = False,
        truncation: Literal["head", "tail"] = "tail",
        add_cls: bool = False,
        ) -> Dict[str, Union[list, "torch.Tensor"]]:
        
        
        codes = timeline.get_column("code").to_list()
        type_ids = [seq_gen.tokenizer.type2id[tok] for tok in timeline['code_type']]
        
        visits_raw = timeline.get_column('seq_id').to_list()
        visit_index = {}
        vid_seq = []
        next_id = 1
        for v in visits_raw:
            key = v
            if key not in visit_index:
                visit_index[key] = next_id
                next_id += 1
            vid_seq.append(visit_index[key])

        stage_cols = ["out_id", "er_id", "hadm_id", "icustay_id"]
        stage_map = {col: i + 1 for i, col in enumerate(stage_cols)}

        stage_ids = []
        stage_values = timeline.select(stage_cols).to_dict(as_series=False)

        for i in range(timeline.height):
            sid = 0
            for col in stage_cols:
                val = stage_values[col][i]
                if val is not None:
                    sid = stage_map[col]
                    break
            stage_ids.append(sid)
        
        if add_cls and self.tokenizer.cls_token is not None:
            codes = [self.tokenizer.cls_token] + codes
            type_ids = [self.tokenizer.type2id['[CLS]']] + type_ids
            vid_seq = [0] + vid_seq
            stage_ids = [0] + stage_ids

        
        
        for i, tok in enumerate(codes):
            if tok.startswith("TIME-GAP//"):
                vid_seq[i] = 0
                stage_ids[i] = 0
                             
        input_ids = self.tokenizer.encode(codes)
        input_ids = self._truncate(input_ids, max_length, truncation)
        type_ids  = self._truncate(type_ids, max_length, truncation)
        vid_seq   = self._truncate(vid_seq, max_length, truncation)
        stage_ids = self._truncate(stage_ids, max_length, truncation)
        attention_mask = [1] * len(input_ids)

        if pad_to_max and max_length is not None and len(input_ids) < max_length:
            pad_len = max_length - len(input_ids)
            type_ids = type_ids + [0] * pad_len
            pad_id = self.tokenizer.pad_id if self.tokenizer.pad_id is not None else 0
            input_ids     = input_ids + [pad_id] * pad_len
            attention_mask = attention_mask + [0] * pad_len
            vid_seq        = vid_seq + [0] * pad_len
            stage_ids      = stage_ids + [0] * pad_len
        
        value_payload = self.build_value_streams(timeline=timeline,
                                                  codes=codes,                     
                                                  add_cls=add_cls and (self.tokenizer.cls_token is not None),
                                                  max_length=max_length,
                                                  pad_to_max=pad_to_max,
                                                  truncation=truncation,
                                                  return_numeric=self.return_numeric,
                                                  return_text=self.return_text)
        
        out = {"input_ids": input_ids, "attention_mask": attention_mask,
               "visit_ids": vid_seq, "stage_ids": stage_ids, "type_ids": type_ids}


        out.update(value_payload)
        return out
    

    
    def build_value_streams(self,
                            timeline: pl.DataFrame,
                            codes: list[str],                 
                            add_cls: bool,
                            max_length: Optional[int],
                            pad_to_max: bool,
                            truncation: str,
                            return_numeric: bool,
                            return_text: bool,
                            ) -> dict:

        if not (return_numeric or return_text):
            return {}

        n = len(codes) - (1 if add_cls else 0)
        has_num  = "numeric_value" in timeline.columns
        has_text = "text_value"   in timeline.columns
        raw_num  = timeline.get_column("numeric_value").to_list() if has_num else [None] * n
        raw_txt  = timeline.get_column("text_value").to_list()    if has_text else [None] * n

        if add_cls:
            raw_num = [None] + raw_num
            raw_txt = [None] + raw_txt
            
        raw_txt = [None if x == '___' else x for x in raw_txt]
        out = {}
        if return_numeric:
            numeric_mask   = [1 if v is not None else 0 for v in raw_num]
            numeric_values = self._truncate(raw_num, max_length, truncation)
            numeric_mask   = self._truncate(numeric_mask, max_length, truncation)
            
            numeric_values = [0.0 if x == None else x for x in numeric_values]
            
            if pad_to_max and max_length is not None and len(numeric_values) < max_length:
                pad_len = max_length - len(numeric_values)
                numeric_values += [0.0]*pad_len
                numeric_mask   += [0] * pad_len
            out["numeric_values"] = numeric_values
            out["numeric_mask"]   = numeric_mask

        if return_text:
            text_mask   = [1 if (t is not None and str(t) != "") else 0 for t in raw_txt]
            text_values = self._truncate(raw_txt, max_length, truncation)
            text_mask   = self._truncate(text_mask,   max_length, truncation)
            text_values = ['' if x == None else x for x in text_values]
            if pad_to_max and max_length is not None and len(text_values) < max_length:
                pad_len = max_length - len(text_values)
                if text_values and isinstance(text_values[0], list):
                    text_values += [[] for _ in range(pad_len)]
                elif text_values and isinstance(text_values[0], dict):
                    text_values += [{} for _ in range(pad_len)]
                else:
                    text_values += ["" for _ in range(pad_len)]
                text_mask += [0]* pad_len
            out["text_values"] = text_values
            out["text_mask"]   = text_mask

        return out
    
  
    def get_overlapped_chunks(self,
        timeline: Dict[str, Iterable],
        chunk_length: Optional[int] = None,
        overlap: Optional[int] = None,
        add_cls_per_chunk: bool = True) -> List[Dict[str, List[Any]]]:
        
        if chunk_length is None or overlap is None:
            chunk_length = self.chunk_length
            overlap = self.overlap

        fields = ["input_ids", "attention_mask", "visit_ids", "stage_ids", "type_ids"]
        for extra in ("numeric_values", "numeric_mask", "text_values", "text_mask"):
            if extra in timeline and extra not in fields:
                fields.append(extra)

        n = len(timeline["input_ids"])

        payload = chunk_length - (1 if add_cls_per_chunk else 0)
        step = payload - overlap

        num_chunks = 1 if n <= payload else ceil((n - payload) / step) + 1
        starts = [i * step for i in range(num_chunks)]

        cls_defaults = {"input_ids": self.tokenizer.cls_id, "attention_mask": 1,"visit_ids": 0, 
                        "stage_ids": 0, "type_ids": self.tokenizer.type2id['[CLS]'], 
                        "numeric_values": 0.0,"numeric_mask": 0, "text_values": "", "text_mask": 0}

        chunks = []
        for start in starts:
            end = min(n, start + payload)
            sliced = {k: timeline[k][start:end] for k in timeline.keys()}

            if "attention_mask" in sliced:
                sliced["attention_mask"] = [1] * len(sliced["input_ids"])

            if add_cls_per_chunk:
                for k in list(sliced.keys()):
                    sliced[k] = [cls_defaults[k]] + sliced[k]

            cur_len = len(sliced["input_ids"])
            if cur_len < chunk_length:
                pad_len = chunk_length - cur_len
                for k in list(sliced.keys()):
                    sliced[k] = self._pad_list(sliced[k], pad_len, 0)

            chunks.append(sliced)
        return chunks


    def _truncate(self, 
                  seq: list[int],
                  max_length,
                  truncation) -> list[int]:
        if max_length is None or len(seq) <= max_length:
            return seq
        return seq[-max_length:] if truncation == "head" else seq[:max_length]
    
    
    def _return_tensors(self, 
                        out) -> dict :
        if type(out) == dict:
            for k in out.keys():
                out[k] = torch.tensor(out[k], dtype=torch.long)
            return out
        
        elif type(out) == list:
            for ch in out:
                for k, v in ch.items():
                    ch[k] = torch.tensor(v, dtype=torch.long)
            return out

    
    def _pad_list(self, lst, pad_len, pad_value):
        if pad_len <= 0:
            return lst
        return lst + [pad_value] * pad_len
    

In [63]:
# for file in tqdm(os.listdir('/scratch/sas10092/ehr-foundation/meds_binned/data/train/')):
    
#     in_fp = f'/scratch/sas10092/ehr-foundation/meds_binned/data/train/{file}'
#     out_fp = f'/scratch/sas10092/ehr-foundation/meds_cat_val/data/train/{file}'
#     transform_shard(in_fp,out_fp)

In [ ]:
# for file in tqdm(os.listdir('../data/meds_normalized/data/train/')):
#     normalized = pd.read_parquet(f'../data/meds_normalized/data/train/{file}')
#     outliers = pd.read_parquet(f'../data/meds_outliers/data/train/{file}')
#     outliers['numeric_value'] = normalized['numeric_value']
#     outliers.to_parquet(f'../data/meds_normalized/data/train/{file}')

In [ ]:
# import polars as pl, re

# in_fp  = "/path/to/input.parquet"
# out_fp = "/path/to/output.parquet"

# df_in  = a0
# df_out = a

# has_bin_re = r"//value_\[.*\)$"

# # 1) Expected base codes from input (trim if they had a bin suffix)
# dfi_base = (
#     df_in
#     .with_columns(
#         pl.when(pl.col("code").str.contains(has_bin_re))
#           .then(pl.col("code").str.replace(r"//value_\[.*\)$", ""))
#           .otherwise(pl.col("code"))
#           .alias("code_expected")
#     )
#     .select("code_expected")
# )

# # 2) Actual base rows from output (exclude the VALUE-BIN rows)
# dfout_base = (
#     df_out
#     .filter(~pl.col("code").str.starts_with("VALUE-BIN//"))
#     .select(pl.col("code").alias("code_actual"))
# )

# # 3) Counts must match (each binned input row expands to 2 output rows, but we filtered the bin rows)
# assert dfi_base.height == dfout_base.height, (
#     dfi_base.height, dfout_base.height
# )

# # 4) Codes must match exactly in order
# mismatch = (
#     dfi_base
#     .with_row_index()
#     .join(dfout_base.with_row_index(), on="index")
#     .filter(pl.col("code_expected") != pl.col("code_actual"))
# )

# assert mismatch.height == 0, mismatch.head()
# print("Base-row codes match ✅")